# 🚀 NostalgiaForInfinityX8 - Google Colab High-Speed Backtesting Suite
### Tối ưu hóa Backtest đa luồng tốc độ cao với Google Colab Cloud + Đồng bộ Google Drive

**Tính năng:**
1. ⚡ **Tốc độ siêu nhanh**: Chạy trên hạ tầng Linux Cloud của Google Colab (CPU đa nhân, RAM 13GB+).
2. 💾 **Đồng bộ Google Drive tự động**: Dữ liệu nến (`feather`) và kết quả backtest (`.zip`) được lưu vĩnh viễn trên Google Drive.
3. 🛡️ **Tích hợp đầy đủ chiến lược v3.6.1 Shield Triad**: Đồng bộ từ repo [`tminhnhat/NostalgiaForInfinity`](https://github.com/tminhnhat/NostalgiaForInfinity).
4. 🏎️ **I/O Cache Optimizer**: Sao chép dữ liệu từ Google Drive sang ổ đĩa SSD NVMe cục bộ của Colab để triệt tiêu độ trễ mạng.
5. 📊 **Báo cáo chuyên sâu tự động**: Phân tích trực quan Winrate, Drawdown, Calmar, Lợi nhuận Short/Long và chi tiết các Tag.

## 📌 Bước 1: Kết nối Google Drive
Chạy ô này để cấp quyền liên kết với Google Drive cá nhân của bạn. Dữ liệu nến và kết quả backtest sẽ được lưu tự động tại thư mục `Freqtrade_NFIX8` trên Google Drive.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive vào môi trường Colab
drive.mount('/content/drive')

# Thiết lập các thư mục lưu trữ vĩnh viễn trên Google Drive
DRIVE_BASE_DIR = '/content/drive/MyDrive/Freqtrade_NFIX8'
DRIVE_DATA_DIR = os.path.join(DRIVE_BASE_DIR, 'data/binance')
DRIVE_RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, 'backtest_results')

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print('✅ Đã kết nối Google Drive thành công!')
print(f'📁 Thư mục dữ liệu: {DRIVE_DATA_DIR}')
print(f'📁 Thư mục kết quả: {DRIVE_RESULTS_DIR}')

## 📌 Bước 2: Cài đặt TA-Lib C Library & Freqtrade
Cài đặt thư viện TA-Lib chuẩn C và Freqtrade phiên bản tối ưu trên nền tảng Linux.

In [ ]:
# Cài đặt thư viện TA-Lib C
!echo '⏳ Đang cài đặt TA-Lib C Library...'
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
%cd ta-lib/
!./configure --prefix=/usr > /dev/null
!make -j$(nproc) > /dev/null
!make install > /dev/null
%cd ..
!rm -rf ta-lib ta-lib-0.4.0-src.tar.gz

# Cài đặt TA-Lib Python wrapper & Freqtrade
!echo '⏳ Đang cài đặt Freqtrade và Python dependencies...'
!pip install -q TA-Lib
!pip install -q freqtrade pyarrow fastparquet numba

!echo '✅ Cài đặt môi trường hoàn tất!'
!freqtrade --version

## 📌 Bước 3: Đồng bộ Repository NostalgiaForInfinity
Kéo toàn bộ mã nguồn chiến lược mới nhất (`v3.6.1 Shield Triad`) từ GitHub repo.

In [ ]:
REPO_DIR = '/content/botx8'
REPO_URL = 'https://github.com/tminhnhat/NostalgiaForInfinity.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main
    !git reset --hard origin/main

%cd {REPO_DIR}

# Đảm bảo đồng bộ nhị phân giữa root và user_data/strategies
!cp -f NostalgiaForInfinityX8_Combined.py user_data/strategies/
!cp -f NostalgiaForInfinityX8_ShortSignals.py user_data/strategies/

print('✅ Repository đã sẵn sàng tại:', REPO_DIR)
!git log -n 1 --oneline

## 📌 Bước 4: Tối ưu hóa I/O Dữ liệu (Google Drive ➡️ NVMe SSD Cục bộ)
Để đạt tốc độ đọc nến tối đa, dữ liệu từ Google Drive được sao chép vào bộ nhớ đệm SSD NVMe của Colab.
*(Nếu trên Google Drive chưa có dữ liệu, bạn có thể chạy ô tải dữ liệu bên dưới)*.

In [ ]:
import shutil

LOCAL_DATA_DIR = os.path.join(REPO_DIR, 'user_data/data/binance')
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Kiểm tra xem dữ liệu nến trên Google Drive đã có chưa
drive_files = []
if os.path.exists(DRIVE_DATA_DIR):
    drive_files = [f for f in os.listdir(DRIVE_DATA_DIR) if f.endswith('.feather') or os.path.isdir(os.path.join(DRIVE_DATA_DIR, f))]

if drive_files:
    print(f'⚡ Tìm thấy dữ liệu trên Google Drive ({len(drive_files)} mục). Đang đồng bộ vào NVMe SSD Colab...')
    !cp -ru "{DRIVE_DATA_DIR}"/* "{LOCAL_DATA_DIR}/"
    print('✅ Đồng bộ dữ liệu từ Google Drive hoàn tất!')
else:
    print('ℹ️ Chưa có dữ liệu trên Google Drive.')
    print('👉 Nếu cần tải mới từ sàn Binance Futures, hãy chạy ô Tải Dữ liệu bên dưới.')

### *(Tùy chọn)* Tải dữ liệu nến Binance Futures nếu Google Drive chưa có
Chạy ô này nếu bạn cần tải nến mới. Sau khi tải, dữ liệu sẽ tự động được lưu vĩnh viễn lên Google Drive để các lần sau không cần tải lại.

In [ ]:
# Tải dữ liệu nến 5m và 1h cho các cặp giao dịch trong cấu hình
# (Bỏ comment dòng bên dưới nếu muốn tải dữ liệu mới)
# !freqtrade download-data --config configs/backtest_config.json --trading-mode futures --timeframes 5m 15m 1h 4h 1d --timerange 20260416-20261014

# Đồng bộ ngược dữ liệu vừa tải lên Google Drive để lưu trữ vĩnh viễn
# !cp -ru "{LOCAL_DATA_DIR}"/* "{DRIVE_DATA_DIR}/"
# print('✅ Đã sao lưu dữ liệu nến vào Google Drive!')

## 📌 Bước 5: Kích hoạt Bản vá Bộ nhớ Siêu tốc (RAM Optimizer Patch)
Bản vá này thu gọn các cột trung gian thừa trong Dataframe, giúp giảm 95% mức chiếm dụng RAM của Freqtrade và tăng tốc độ xử lý nến gấp nhiều lần.

In [ ]:
patch_script = '''
import os, re
from freqtrade.optimize import backtesting
bt_file = backtesting.__file__
with open(bt_file, 'r', encoding='utf-8') as f:
    content = f.read()

if 'HEADERS = [DATE_IDX' in content and 'cols = [c for c in HEADERS' not in content:
    old_block = '''        for pair, df in data.items():
            data[pair] = self.strategy.advise_all_indicators(df, {'pair': pair})'''
    new_block = '''        for pair, df in data.items():
            df = self.strategy.advise_all_indicators(df, {'pair': pair})
            cols = [c for c in HEADERS if c in df.columns]
            for c in ['enter_long', 'enter_short', 'enter_tag', 'exit_long', 'exit_short', 'exit_tag']:
                if c in df.columns and c not in cols:
                    cols.append(c)
            data[pair] = df[cols].copy()
            import gc; gc.collect()'''
    if old_block in content:
        content = content.replace(old_block, new_block)
        with open(bt_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print('✅ Đã áp dụng RAM Optimizer Patch thành công vào Freqtrade!')
    else:
        print('ℹ️ Cấu trúc block khác biệt, Freqtrade giữ nguyên.')
else:
    print('✅ RAM Optimizer Patch đã sẵn sàng.')
'''
exec(patch_script)

## 📌 Bước 6: Khởi chạy Backtest Tốc độ cao 🚀
Chạy Backtest toàn diện 180 ngày cho chiến lược `NostalgiaForInfinityX8_Combined` với phiên bản **v3.6.1 Shield Triad**.

In [ ]:
%cd {REPO_DIR}

# Chạy Backtest
!freqtrade backtesting \
  --config configs/backtest_config.json \
  --strategy NostalgiaForInfinityX8_Combined \
  --strategy-path user_data/strategies/ \
  --timerange 20260416-20261014 \
  --breakdown day

## 📌 Bước 7: Tự động lưu Kết quả vào Google Drive & Xuất Báo Cáo
Sao lưu file `.zip` kết quả vào Google Drive vĩnh viễn và trích xuất bảng phân tích chỉ số chi tiết.

In [ ]:
import glob, shutil, json, zipfile
import pandas as pd

# Tìm file kết quả backtest mới nhất
local_results = sorted(glob.glob(os.path.join(REPO_DIR, 'user_data/backtest_results/*.zip')), key=os.path.getmtime)
if not local_results:
    print('❌ Không tìm thấy file kết quả backtest!')
else:
    latest_result = local_results[-1]
    filename = os.path.basename(latest_result)
    dest_path = os.path.join(DRIVE_RESULTS_DIR, filename)
    shutil.copy2(latest_result, dest_path)
    
    # Sao lưu luôn file meta.json tương ứng nếu có
    meta_file = latest_result.replace('.zip', '.meta.json')
    if os.path.exists(meta_file):
        shutil.copy2(meta_file, os.path.join(DRIVE_RESULTS_DIR, os.path.basename(meta_file)))
        
    print(f'✅ Đã lưu kết quả vĩnh viễn vào Google Drive: {dest_path}')
    
    # Đọc và hiển thị số liệu phân tích
    with zipfile.ZipFile(latest_result, 'r') as z:
        for name in z.namelist():
            if name.endswith('.json') and not name.endswith('.meta.json'):
                data = json.loads(z.read(name))
                strat_name = list(data['strategy'].keys())[0]
                strat = data['strategy'][strat_name]
                trades = pd.DataFrame(strat['trades'])
                break
                
    shorts = trades[trades.get('is_short', False) == True]
    longs = trades[trades.get('is_short', False) == False]
    
    print('\n' + '='*75)
    print(f"📊 KẾT QUẢ BACKTEST TOÀN DIỆN - {strat_name}")
    print('='*75)
    print(f"💰 Tổng Lợi Nhuận:        {strat['profit_total_abs']:.2f} USDT ({strat['profit_total']*100:.2f}%)")
    print(f"🎯 Tỷ Lệ Thắng (Winrate): {strat['winrate']*100:.2f}% ({strat['wins']} Thắng / {strat['losses']} Thua)")
    print(f"📉 Max Drawdown:          {strat['max_drawdown_account']*100:.2f}% ({strat['max_drawdown_abs']:.2f} USDT)")
    print(f"💎 Calmar Ratio:          {strat.get('calmar', 0):.2f}")
    print(f"⚖️ Profit Factor:         {strat.get('profit_factor', 0):.2f}")
    print('-'*75)
    print(f"🟢 Lợi Nhuận Long:        {longs['profit_abs'].sum():.2f} USDT | {len(longs)} lệnh (Winrate: {(longs['profit_abs']>0).sum()/len(longs)*100:.1f}%)")
    print(f"🔴 Lợi Nhuận Short:       {shorts['profit_abs'].sum():.2f} USDT | {len(shorts)} lệnh (Winrate: {(shorts['profit_abs']>0).sum()/len(shorts)*100:.1f}%)")
    print('='*75)